# Sample Downstream Responses — V2 Cloud Run

Look at concrete examples of each pretrained LM's behaviour on SciQ, ARC-Easy, and XNLI, and see *why* the headline accuracy numbers move the way they do.

Data source: `results_pi/downstream/{model}/{paper_tasks,xnli}/.../samples_*.jsonl`  
Models: 6 LMs trained on 3B mC4 tokens with the V2 tokenizers (cloud Stage 2 run, May 2026).

**TL;DR for this notebook**
1. SciQ "accuracy" is a positional artifact — the correct answer is always at index 3, BPE-family LMs prefer index 3, Unigram-family avoid it. Neither family is doing science reasoning.
2. ARC-Easy (positions randomized) shows all 6 LMs near random.
3. XNLI shows the same near-chance behaviour with small spread differences across models.

In [5]:
from __future__ import annotations

import json
import statistics as st
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

# Project root — assume notebook lives in <repo>/notebooks/
ROOT = Path('..').resolve()
RESULTS = ROOT / 'results_pi' / 'downstream'

# All six models in canonical order.
MODELS = ['bpe', 'parity_bpe', 'unigram', 'adat', 'paat_a10', 'paat_a100_l0']

assert RESULTS.exists(), f'expected {RESULTS} to exist — pull cloud results first'
print('models found:')
for m in MODELS:
    have = (RESULTS / m).exists()
    print(f'  {"✓" if have else "✗"} {m}')

models found:
  ✓ bpe
  ✓ parity_bpe
  ✓ unigram
  ✓ adat
  ✓ paat_a10
  ✓ paat_a100_l0


In [6]:
def load_samples(model: str, task: str) -> list[dict]:
    """Load all per-example records for one (model, task) pair.

    Each row of the JSONL is one example. Order is identical across
    models because lm-eval-harness iterates the dataset deterministically.
    """
    sub = 'paper_tasks' if task in ('sciq', 'arc_easy') else 'xnli'
    pattern = f'samples_{task}_*.jsonl'
    files = list((RESULTS / model / sub).rglob(pattern))
    if not files:
        return []
    # Pick the most recent (lm-eval can write multiple if re-run).
    f = max(files, key=lambda p: p.stat().st_mtime)
    out = []
    with f.open() as fh:
        for line in fh:
            out.append(json.loads(line))
    return out


def model_pred(record: dict) -> tuple[int, list[float]]:
    """Return (predicted_choice_index, [loglik per choice]).

    lm-eval stores responses in `filtered_resps` (after any filter
    pipeline) or `resps` (raw).  Each element is `[loglikelihood,
    is_greedy_match]`.
    """
    resps = record.get('filtered_resps') or record.get('resps') or []
    logls = [float(r[0] if isinstance(r, list) else r) for r in resps]
    pred = int(max(range(len(logls)), key=lambda i: logls[i])) if logls else -1
    return pred, logls


# Sanity: confirm shape
_test = load_samples('bpe', 'sciq')
print(f'bpe SciQ: {len(_test)} examples')
print(f'first record keys: {sorted(_test[0].keys())[:10]}…')

bpe SciQ: 1000 examples
first record keys: ['acc', 'acc_norm', 'arguments', 'doc', 'doc_hash', 'doc_id', 'filter', 'filtered_resps', 'metrics', 'prompt_hash']…


## 1. SciQ — same example across all 6 models

For SciQ, lm-eval-harness presents the choices as `[distractor1, distractor2, distractor3, correct_answer]` — **the correct answer is always at index 3**.

We show the same example for all 6 LMs side-by-side: the question, the four candidate answers, the loglikelihood the LM assigned to each, the LM's pick, and whether it was right.

In [7]:
def render_sciq_row(records: dict[str, list[dict]], example_idx: int) -> str:
    """Render one SciQ example across all models as a markdown block."""
    ref = records[MODELS[0]][example_idx]
    doc = ref['doc']
    question = doc['question']
    choices = [doc['distractor1'], doc['distractor2'], doc['distractor3'], doc['correct_answer']]
    target = int(ref['target'])

    lines = [f'### Q{example_idx}: {question}', '']
    if doc.get('support'):
        sup = doc['support'].strip().replace('\n', ' ')
        if len(sup) > 200:
            sup = sup[:200] + '…'
        lines.append(f'_Support_: {sup}')
        lines.append('')
    lines.append(f'**Choices** (correct = idx {target}: `{choices[target]}`)')
    lines.append('')
    for i, c in enumerate(choices):
        marker = ' ← CORRECT' if i == target else ''
        lines.append(f'- `[{i}]` {c}{marker}')
    lines.append('')

    # Per-model table.
    lines.append('| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |')
    lines.append('|---|---:|---:|---:|---:|:---:|:---:|')
    for m in MODELS:
        rec = records[m][example_idx]
        pred, logls = model_pred(rec)
        is_correct = pred == target
        cells = []
        for i in range(4):
            v = f'{logls[i]:.2f}'
            if i == pred:
                v = f'**{v}**'
            cells.append(v)
        mark = '✓' if is_correct else '✗'
        lines.append(f'| `{m}` | ' + ' | '.join(cells) + f' | `{pred}` | {mark} |')
    lines.append('')
    return '\n'.join(lines)


# Load SciQ for all models
sciq = {m: load_samples(m, 'sciq') for m in MODELS}
n_sciq = len(sciq[MODELS[0]])
print(f'SciQ examples per model: {n_sciq}')

# Show 3 examples — early, middle, near-end
for idx in [0, n_sciq // 2, n_sciq - 1]:
    display(Markdown(render_sciq_row(sciq, idx)))

SciQ examples per model: 1000


### Q0: Compounds that are capable of accepting electrons, such as o 2 or f2, are called what?

_Support_: Oxidants and Reductants Compounds that are capable of accepting electrons, such as O 2 or F2, are calledoxidants (or oxidizing agents) because they can oxidize other compounds. In the process of accep…

**Choices** (correct = idx 3: `oxidants`)

- `[0]` antioxidants
- `[1]` Oxygen
- `[2]` residues
- `[3]` oxidants ← CORRECT

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---:|---:|---:|---:|:---:|:---:|
| `bpe` | -10.75 | -11.88 | -20.38 | **-8.12** | `3` | ✓ |
| `parity_bpe` | -8.38 | -13.88 | -16.25 | **-7.03** | `3` | ✓ |
| `unigram` | **-32.00** | -35.75 | -33.50 | -32.00 | `0` | ✗ |
| `adat` | -40.00 | **-33.75** | -36.75 | -40.00 | `1` | ✗ |
| `paat_a10` | -33.75 | **-33.00** | -36.00 | -33.75 | `1` | ✗ |
| `paat_a100_l0` | -30.25 | **-29.25** | -36.00 | -30.25 | `1` | ✗ |


### Q500: Invertebrates (and higher animals) can also be placed in one of two groups based on how they develop as what?

_Support_: Eight invertebrate phyla contain most invertebrate species. Invertebrates (and higher animals) can also be placed in one of two groups based on how they develop as embryos.

**Choices** (correct = idx 3: `embryos`)

- `[0]` chromosomes
- `[1]` hormones
- `[2]` cells
- `[3]` embryos ← CORRECT

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---:|---:|---:|---:|:---:|:---:|
| `bpe` | -18.88 | -14.50 | -10.81 | **-10.25** | `3` | ✓ |
| `parity_bpe` | -18.50 | -17.50 | **-12.38** | -17.88 | `2` | ✗ |
| `unigram` | -38.75 | **-21.25** | -28.88 | -28.75 | `1` | ✗ |
| `adat` | -38.50 | -26.25 | **-22.50** | -34.75 | `2` | ✗ |
| `paat_a10` | -39.50 | -24.75 | **-22.00** | -33.00 | `2` | ✗ |
| `paat_a100_l0` | -38.50 | -24.50 | **-20.50** | -33.50 | `2` | ✗ |


### Q999: In what form is atmospheric sulfur found?

_Support_: On land, sulfur is deposited in four major ways: precipitation, direct fallout from the atmosphere, rock weathering, and geothermal vents (Figure 20.17). Atmospheric sulfur is found in the form of sul…

**Choices** (correct = idx 3: `sulfur dioxide (so2)`)

- `[0]` formaldehyde
- `[1]` sulfuric acid
- `[2]` sulfur monoxide
- `[3]` sulfur dioxide (so2) ← CORRECT

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---:|---:|---:|---:|:---:|:---:|
| `bpe` | -17.88 | **-17.62** | -18.75 | -24.62 | `1` | ✗ |
| `parity_bpe` | **-12.50** | -15.81 | -21.12 | -32.25 | `0` | ✗ |
| `unigram` | -40.50 | **-37.00** | -44.50 | -51.75 | `1` | ✗ |
| `adat` | **-35.50** | -43.50 | -46.50 | -53.50 | `0` | ✗ |
| `paat_a10` | -44.00 | **-36.25** | -41.25 | -52.00 | `1` | ✗ |
| `paat_a100_l0` | **-32.25** | -34.50 | -42.75 | -48.00 | `0` | ✗ |


### Position bias on SciQ — the headline finding

The 40-pp SciQ accuracy gap between BPE-family and Unigram-family is **entirely positional**, not a real reasoning gap. Below: how often each model picks each choice index, alongside its accuracy.

In [8]:
rows = []
for m in MODELS:
    chose = Counter()
    correct = 0
    for r in sciq[m]:
        pred, _ = model_pred(r)
        chose[pred] += 1
        if pred == int(r['target']):
            correct += 1
    n = len(sciq[m])
    rows.append({
        'model':    m,
        'acc %':    f'{100 * correct / n:.1f}',
        'pick[0] %': f'{100 * chose[0] / n:.1f}',
        'pick[1] %': f'{100 * chose[1] / n:.1f}',
        'pick[2] %': f'{100 * chose[2] / n:.1f}',
        'pick[3] %': f'{100 * chose[3] / n:.1f}',
    })
df = pd.DataFrame(rows)
df

,model,acc %,pick[0] %,pick[1] %,pick[2] %,pick[3] %
0,bpe,67.7,10.8,10.7,10.8,67.7
1,parity_bpe,69.9,10.4,9.2,10.5,69.9
2,unigram,27.5,27.2,22.2,23.1,27.5
3,adat,24.4,28.8,25.2,21.6,24.4
4,paat_a10,25.8,26.7,24.5,23.0,25.8
5,paat_a100_l0,27.2,27.7,23.5,21.6,27.2


**Reading this table:** correct answer is always at index 3, so `acc %` ≈ `pick[3] %`. The Unigram-family LMs show a strong *anti*-position-3 bias (~13% picks); BPE-family shows a *pro*-position-3 bias (~44% picks). Neither pattern indicates real comprehension.

## 2. ARC-Easy — choice positions randomized in source data

Unlike SciQ, ARC's correct answer position is genuinely shuffled. Position bias should average out, so any signal here is real.

In [9]:
def render_arc_row(records: dict[str, list[dict]], example_idx: int) -> str:
    ref = records[MODELS[0]][example_idx]
    doc = ref['doc']
    question = doc.get('question', '<no question>')
    # ARC structure: doc['choices'] = {'text': [...], 'label': ['A','B','C','D']}
    ch = doc.get('choices', {})
    texts = ch.get('text', [])
    labels = ch.get('label', [str(i) for i in range(len(texts))])
    answer_key = doc.get('answerKey', None)
    target = int(ref['target'])

    lines = [f'### Q{example_idx}: {question}', '']
    lines.append(f'**Choices** (correct = idx {target}, label `{answer_key}`)')
    for i, (lbl, txt) in enumerate(zip(labels, texts)):
        marker = ' ← CORRECT' if i == target else ''
        lines.append(f'- `[{i}]` ({lbl}) {txt}{marker}')
    lines.append('')
    lines.append('| Model | ' + ' | '.join(f'logl[{i}]' for i in range(len(texts))) + ' | Pick | ✓? |')
    lines.append('|---' * (len(texts) + 3) + '|')
    for m in MODELS:
        rec = records[m][example_idx]
        pred, logls = model_pred(rec)
        cells = []
        for i in range(len(logls)):
            v = f'{logls[i]:.2f}'
            if i == pred:
                v = f'**{v}**'
            cells.append(v)
        mark = '✓' if pred == target else '✗'
        lines.append(f'| `{m}` | ' + ' | '.join(cells) + f' | `{pred}` | {mark} |')
    lines.append('')
    return '\n'.join(lines)


arc = {m: load_samples(m, 'arc_easy') for m in MODELS}
n_arc = len(arc[MODELS[0]])
print(f'ARC-Easy examples per model: {n_arc}')
for idx in [0, n_arc // 2, n_arc - 1]:
    display(Markdown(render_arc_row(arc, idx)))

ARC-Easy examples per model: 2376


### Q0: Which statement best explains why photosynthesis is the foundation of most food webs?

**Choices** (correct = idx 0, label `A`)
- `[0]` (A) Sunlight is the source of energy for nearly all ecosystems. ← CORRECT
- `[1]` (B) Most ecosystems are found on land instead of in water.
- `[2]` (C) Carbon dioxide is more available than other gases.
- `[3]` (D) The producers in all ecosystems are plants.

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---|---|---|---|---|---|
| `bpe` | -70.00 | -64.50 | -61.50 | **-55.50** | `3` | ✗ |
| `parity_bpe` | -66.50 | -69.00 | -60.75 | **-54.00** | `3` | ✗ |
| `unigram` | -85.00 | -83.00 | -72.00 | **-70.50** | `3` | ✗ |
| `adat` | -82.00 | -86.50 | -69.50 | **-68.50** | `3` | ✗ |
| `paat_a10` | -84.50 | -78.00 | -72.00 | **-69.50** | `3` | ✗ |
| `paat_a100_l0` | -87.00 | -83.50 | **-69.50** | -72.50 | `2` | ✗ |


### Q1188: Which current property of our solar system is mainly due to solar emissions occurring soon after the initiation of fusion in the Sun's core?

**Choices** (correct = idx 3, label `D`)
- `[0]` (A) directions of revolution of the planets
- `[1]` (B) directions of rotation of the planets
- `[2]` (C) distribution of moons among the planets
- `[3]` (D) distribution of elements among the planets ← CORRECT

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---|---|---|---|---|---|
| `bpe` | **-38.00** | -40.00 | -46.75 | -40.00 | `0` | ✗ |
| `parity_bpe` | -39.50 | -43.00 | -42.50 | **-38.25** | `3` | ✓ |
| `unigram` | **-60.25** | -62.50 | -72.00 | -62.00 | `0` | ✗ |
| `adat` | -56.50 | **-54.00** | -62.75 | -56.50 | `1` | ✗ |
| `paat_a10` | -59.75 | **-59.50** | -69.00 | -64.50 | `1` | ✗ |
| `paat_a100_l0` | **-54.25** | -54.25 | -63.00 | -58.75 | `0` | ✗ |


### Q2375: Which part of a sunflower plant absorbs water and nutrients?

**Choices** (correct = idx 0, label `A`)
- `[0]` (A) Roots ← CORRECT
- `[1]` (B) Stems
- `[2]` (C) Leaves
- `[3]` (D) Flowers

| Model | logl[0] | logl[1] | logl[2] | logl[3] | Pick | ✓? |
|---|---|---|---|---|---|---|
| `bpe` | -14.44 | -15.12 | -12.69 | **-11.31** | `3` | ✗ |
| `parity_bpe` | -13.00 | -16.75 | -13.25 | **-12.06** | `3` | ✗ |
| `unigram` | -30.62 | -26.25 | **-20.12** | -20.12 | `2` | ✗ |
| `adat` | -28.75 | -27.38 | **-20.75** | -20.75 | `2` | ✗ |
| `paat_a10` | -28.75 | -26.50 | **-21.75** | -21.75 | `2` | ✗ |
| `paat_a100_l0` | -28.00 | -26.75 | **-17.88** | -17.88 | `2` | ✗ |


In [10]:
# Position-pick distribution for ARC — should be more uniform than SciQ if
# answer positions are genuinely shuffled.
rows = []
for m in MODELS:
    chose = Counter()
    correct = 0
    n_choices_seen = 0
    for r in arc[m]:
        pred, logls = model_pred(r)
        chose[pred] += 1
        n_choices_seen = max(n_choices_seen, len(logls))
        if pred == int(r['target']):
            correct += 1
    n = len(arc[m])
    row = {'model': m, 'acc %': f'{100 * correct / n:.1f}'}
    for i in range(n_choices_seen):
        row[f'pick[{i}] %'] = f'{100 * chose[i] / n:.1f}'
    rows.append(row)
pd.DataFrame(rows)

,model,acc %,pick[0] %,pick[1] %,pick[2] %,pick[3] %,pick[4] %
0,bpe,28.8,36.1,25.4,19.4,19.1,0.0
1,parity_bpe,27.3,38.1,24.6,18.5,18.8,0.0
2,unigram,24.8,37.9,26.3,18.4,17.4,0.0
3,adat,25.2,40.4,26.1,17.3,16.2,0.0
4,paat_a10,25.2,40.4,25.2,17.2,17.1,0.0
5,paat_a100_l0,25.1,40.1,25.8,17.2,16.9,0.0


## 3. XNLI — multilingual entailment (3-way)

Each example: a premise + a hypothesis, classified as **entailment / neutral / contradiction**. We show the same English example across models, then sample one from a non-English language.

In [11]:
XNLI_LABELS = ['entailment', 'neutral', 'contradiction']


def render_xnli_row(records: dict[str, list[dict]], example_idx: int, lang: str) -> str:
    ref = records[MODELS[0]][example_idx]
    doc = ref['doc']
    premise = doc.get('premise', '')
    hypothesis = doc.get('hypothesis', '')
    target = int(ref['target'])

    lines = [f'### {lang.upper()} #{example_idx}', '']
    lines.append(f'**Premise**: {premise}')
    lines.append('')
    lines.append(f'**Hypothesis**: {hypothesis}')
    lines.append('')
    lines.append(f'**Gold label**: `[{target}]` {XNLI_LABELS[target]}')
    lines.append('')
    lines.append('| Model | logl(entail) | logl(neutral) | logl(contra) | Pick | ✓? |')
    lines.append('|---|---:|---:|---:|:---:|:---:|')
    for m in MODELS:
        rec = records[m][example_idx]
        pred, logls = model_pred(rec)
        cells = []
        for i in range(len(logls)):
            v = f'{logls[i]:.2f}'
            if i == pred:
                v = f'**{v}**'
            cells.append(v)
        mark = '✓' if pred == target else '✗'
        lines.append(f'| `{m}` | ' + ' | '.join(cells) + f' | {XNLI_LABELS[pred]} | {mark} |')
    lines.append('')
    return '\n'.join(lines)


for lang in ['xnli_en', 'xnli_zh']:
    samples = {m: load_samples(m, lang) for m in MODELS}
    if not samples[MODELS[0]]:
        print(f'no samples found for {lang}')
        continue
    n = len(samples[MODELS[0]])
    for idx in [0, n // 3]:
        display(Markdown(render_xnli_row(samples, idx, lang)))

### XNLI_EN #0

**Premise**: And he said, Mama, I'm home.

**Hypothesis**: He called his mom as soon as the school bus dropped him off.

**Gold label**: `[1]` neutral

| Model | logl(entail) | logl(neutral) | logl(contra) | Pick | ✓? |
|---|---:|---:|---:|:---:|:---:|
| `bpe` | **-159.00** | -159.00 | -160.00 | entailment | ✗ |
| `parity_bpe` | **-167.00** | -168.00 | -167.00 | entailment | ✗ |
| `unigram` | -159.00 | **-158.00** | -159.00 | neutral | ✓ |
| `adat` | **-155.00** | -157.00 | -157.00 | entailment | ✗ |
| `paat_a10` | **-159.00** | -161.00 | -161.00 | entailment | ✗ |
| `paat_a100_l0` | **-157.00** | -157.00 | -159.00 | entailment | ✗ |


### XNLI_EN #830

**Premise**: And though he should dare attempt it, be sure that his own officers will not dare to do other than oppose him.

**Hypothesis**: His officers respect him.

**Gold label**: `[1]` neutral

| Model | logl(entail) | logl(neutral) | logl(contra) | Pick | ✓? |
|---|---:|---:|---:|:---:|:---:|
| `bpe` | **-196.00** | -198.00 | -199.00 | entailment | ✗ |
| `parity_bpe` | **-200.00** | -200.00 | -201.00 | entailment | ✗ |
| `unigram` | **-191.00** | -192.00 | -193.00 | entailment | ✗ |
| `adat` | **-191.00** | -192.00 | -192.00 | entailment | ✗ |
| `paat_a10` | **-194.00** | -197.00 | -197.00 | entailment | ✗ |
| `paat_a100_l0` | **-194.00** | -195.00 | -198.00 | entailment | ✗ |


### XNLI_ZH #0

**Premise**: 他说，妈妈，我回来了。

**Hypothesis**: 校车把他放下后，他立即给他妈妈打了电话。

**Gold label**: `[1]` neutral

| Model | logl(entail) | logl(neutral) | logl(contra) | Pick | ✓? |
|---|---:|---:|---:|:---:|:---:|
| `bpe` | -193.00 | **-187.00** | -194.00 | neutral | ✓ |
| `parity_bpe` | -178.00 | **-170.00** | -180.00 | neutral | ✓ |
| `unigram` | -171.00 | **-166.00** | -172.00 | neutral | ✓ |
| `adat` | -174.00 | **-167.00** | -176.00 | neutral | ✓ |
| `paat_a10` | -173.00 | **-169.00** | -175.00 | neutral | ✓ |
| `paat_a100_l0` | -176.00 | **-171.00** | -177.00 | neutral | ✓ |


### XNLI_ZH #830

**Premise**: 虽然他应该敢于尝试，但要确保他自己的官员不敢做出别的来反对他。

**Hypothesis**: 他的军官尊重他。

**Gold label**: `[1]` neutral

| Model | logl(entail) | logl(neutral) | logl(contra) | Pick | ✓? |
|---|---:|---:|---:|:---:|:---:|
| `bpe` | -227.00 | **-218.00** | -228.00 | neutral | ✓ |
| `parity_bpe` | -221.00 | **-215.00** | -224.00 | neutral | ✓ |
| `unigram` | -214.00 | **-207.00** | -213.00 | neutral | ✓ |
| `adat` | -209.00 | **-201.00** | -210.00 | neutral | ✓ |
| `paat_a10` | -212.00 | **-206.00** | -213.00 | neutral | ✓ |
| `paat_a100_l0` | -209.00 | **-203.00** | -209.00 | neutral | ✓ |


## 4. Cross-task summary

In [12]:
comp = json.loads((RESULTS / 'comparison.json').read_text())
rows = []
for m in MODELS:
    pt = comp[m]['paper_tasks']
    xnli = comp[m]['xnli']
    xs = [v for k, v in xnli.items() if k.startswith('xnli_') and k != 'avg']
    rows.append({
        'model':       m,
        'SciQ %':      pt['sciq'],
        'ARC-E %':     pt['arc_easy'],
        'XNLI mean':   round(st.mean(xs), 2),
        'XNLI std ↓':  round(st.stdev(xs), 3),
        'XNLI range':  f'{min(xs):.2f}–{max(xs):.2f}',
    })
pd.DataFrame(rows)

,model,SciQ %,ARC-E %,XNLI mean,XNLI std ↓,XNLI range
0,bpe,67.7,28.79,34.02,0.900,33.17–35.90
1,parity_bpe,69.9,27.31,33.73,0.713,33.01–35.34
2,unigram,27.5,24.79,33.33,0.401,32.29–33.90
3,adat,24.4,25.17,33.44,0.457,32.61–34.38
4,paat_a10,25.8,25.21,33.46,0.757,32.13–35.10
5,paat_a100_l0,27.2,25.13,33.62,0.537,33.01–34.70


## 5. Conclusions

1. **SciQ numbers are dominated by positional bias.** Drop SciQ from any tokenizer comparison until the eval is fixed (either a randomization wrapper around the lm-eval task or a different benchmark altogether).
2. **ARC-Easy and XNLI mean show all six LMs near random.** At 35 M params / 3 B tokens with only 2.5 % English content, the LMs simply don't have enough English exposure to clear chance on knowledge-based tasks.
3. **XNLI std differences are small at 3 B tokens.** The earlier V1 (1 B token, local) report claimed PAAT cuts XNLI std 4× vs ADAT — at the 3× larger budget that gap shrinks dramatically. The Stage 1 parity gains (Gini(tok/sent) of 0.055 for parity-BPE, 0.119 for paat_a100_l0) are real, but they don't translate into proportional XNLI-std gains at this LM scale.
4. **What to do next** — either move to a continuous evaluation (held-out perplexity on a multilingual eval set), scale to a larger LM (medium ≈ 85 M params or 200 M+), or up-sample English in the pretraining corpus to give knowledge-based tasks a real chance.